In [1]:
import json
import numpy as np
from collections import defaultdict
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics.pairwise import cosine_similarity
import re
import string

In [5]:
!pip install sentence-transformers

Defaulting to user installation because normal site-packages is not writeable
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 493.7/493.7 KB 1.7 MB/s eta 0:00:00a 0:00:01


In [2]:
try:
    from sentence_transformers import SentenceTransformer
    HAS_SENTENCE_TRANSFORMERS = True
except ImportError:
    HAS_SENTENCE_TRANSFORMERS = False
    print("Warning: Install sentence-transformers: pip install sentence-transformers")


/home/aalkan/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# Load datasets accordingly to the subtask

with open("../data/SOMD 2026/subtask 2/train_data.jsonl", 'r') as f:
    train_data = [json.loads(l) for l in list(f)]
    
with open("../data/SOMD 2026/subtask 2/test_data.jsonl", 'r') as f:
    test_data = [json.loads(l) for l in list(f)]
    
with open("../data/SOMD 2026/subtask 2/train_labels.json", "r") as f:
    train_labels = json.load(f)

In [4]:
class MentionDocumentEmbedding:
    """
    Cluster software mentions by embedding mention + document context.
    
    The key insight: mentions referring to the same software should appear
    in similar document contexts (similar research domains, methodologies, etc.)
    """
    
    def __init__(self, mentions_data, embedding_model='all-MiniLM-L6-v2'):
        """
        Args:
            mentions_data: List of mention dictionaries with keys:
                          mention, mention_id, start, end, type, docid, 
                          relations, sentence
            embedding_model: Sentence transformer model name
        """
        if not HAS_SENTENCE_TRANSFORMERS:
            raise ImportError("Install: pip install sentence-transformers")
        
        self.mentions = mentions_data
        self.mention_dict = {m['mention_id']: m for m in mentions_data}
        
        # Group mentions by document
        self.doc_mentions = defaultdict(list)
        for m in mentions_data:
            self.doc_mentions[m['docid']].append(m)
        
        print(f"Loading embedding model: {embedding_model}...")
        self.model = SentenceTransformer(embedding_model)
        print(f"Model loaded! Processing {len(mentions_data)} mentions from {len(self.doc_mentions)} documents")
        
        self.embeddings_cache = {}
    
    def normalize_mention(self, mention_text):
        """Normalize mention text."""
        text = mention_text.lower()
        text = re.sub(r'\s*v\.?\s*\d+[\w\.-]*', '', text)
        text = text.translate(str.maketrans('', '', string.punctuation))
        text = ' '.join(text.split())
        return text
    
    def get_embedding(self, text):
        """Get embedding with caching."""
        if text in self.embeddings_cache:
            return self.embeddings_cache[text]
        
        embedding = self.model.encode(text, convert_to_numpy=True)
        self.embeddings_cache[text] = embedding
        return embedding
    
    def build_document_context(self, docid, max_sentences=10):
        """
        Build document context by aggregating sentences from that document.
        
        Args:
            docid: Document ID
            max_sentences: Maximum sentences to include (for efficiency)
        
        Returns:
            String representing document context
        """
        doc_mentions = self.doc_mentions[docid]
        
        # Collect unique sentences from this document
        sentences = []
        seen_sentences = set()
        
        for mention in doc_mentions:
            sent = mention['sentence']
            if sent not in seen_sentences:
                sentences.append(sent)
                seen_sentences.add(sent)
                if len(sentences) >= max_sentences:
                    break
        
        # Combine sentences as document context
        doc_context = ' '.join(sentences)
        return doc_context
    
    def mention_plus_document_clustering(self, 
                                        distance_threshold=0.4,
                                        mention_weight=0.5,
                                        document_weight=0.5,
                                        linkage='average'):
        """
        Method 1: Embed "mention + full document context" together.
        
        Creates a combined representation by concatenating mention with
        document context, then embedding the whole thing.
        
        Args:
            distance_threshold: Clustering threshold (lower = more clusters)
            mention_weight: Weight for mention in combined text
            document_weight: Weight for document context
            linkage: 'average', 'complete', or 'single'
        """
        print("Building mention + document representations...")
        combined_texts = []
        
        for mention_obj in self.mentions:
            mention_text = self.normalize_mention(mention_obj['mention'])
            doc_context = self.build_document_context(mention_obj['docid'])
            
            # Create weighted combination
            # Repeat mention text to give it more weight
            mention_repeated = ' '.join([mention_text] * int(mention_weight * 10))
            doc_sampled = doc_context[:int(document_weight * 1000)]  # Limit doc length
            
            combined_text = f"{mention_repeated} {doc_sampled}"
            combined_texts.append(combined_text)
        
        print("Encoding combined representations...")
        embeddings = np.array([self.get_embedding(text) for text in combined_texts])
        
        print("Clustering...")
        clustering = AgglomerativeClustering(
            n_clusters=None,
            distance_threshold=distance_threshold,
            linkage=linkage,
            metric='cosine'
        )
        labels = clustering.fit_predict(embeddings)
        
        return self._labels_to_clusters(labels)
    
    def separate_embedding_clustering(self,
                                     distance_threshold=0.4,
                                     mention_weight=0.6,
                                     document_weight=0.4,
                                     linkage='average'):
        """
        Method 2: Embed mention and document separately, then combine.
        
        This gives you more control over the contribution of each component.
        
        Args:
            distance_threshold: Clustering threshold
            mention_weight: Weight for mention embedding (0-1)
            document_weight: Weight for document embedding (0-1)
            linkage: Linkage method
        """
        print("Encoding mentions...")
        mention_texts = [self.normalize_mention(m['mention']) for m in self.mentions]
        mention_embeddings = np.array([self.get_embedding(text) for text in mention_texts])
        
        print("Encoding document contexts...")
        doc_embeddings = []
        for mention_obj in self.mentions:
            doc_context = self.build_document_context(mention_obj['docid'])
            doc_emb = self.get_embedding(doc_context)
            doc_embeddings.append(doc_emb)
        doc_embeddings = np.array(doc_embeddings)
        
        # Normalize embeddings
        mention_embeddings = mention_embeddings / np.linalg.norm(mention_embeddings, axis=1, keepdims=True)
        doc_embeddings = doc_embeddings / np.linalg.norm(doc_embeddings, axis=1, keepdims=True)
        
        # Combine with weights
        print("Combining embeddings...")
        combined_embeddings = (mention_weight * mention_embeddings + 
                              document_weight * doc_embeddings)
        
        print("Clustering...")
        clustering = AgglomerativeClustering(
            n_clusters=None,
            distance_threshold=distance_threshold,
            linkage=linkage,
            metric='cosine'
        )
        labels = clustering.fit_predict(combined_embeddings)
        
        return self._labels_to_clusters(labels)
    
    def mention_sentence_document_clustering(self,
                                            distance_threshold=0.4,
                                            mention_weight=0.4,
                                            sentence_weight=0.3,
                                            document_weight=0.3,
                                            linkage='average'):
        """
        Method 3: Three-level embedding - mention + sentence + document.
        
        Most granular approach:
        - Mention: The software name itself
        - Sentence: Immediate context where mention appears
        - Document: Broader document context
        
        Args:
            distance_threshold: Clustering threshold
            mention_weight: Weight for mention embedding
            sentence_weight: Weight for sentence embedding
            document_weight: Weight for document embedding
            linkage: Linkage method
        """
        print("Encoding mentions...")
        mention_texts = [self.normalize_mention(m['mention']) for m in self.mentions]
        mention_embeddings = np.array([self.get_embedding(text) for text in mention_texts])
        
        print("Encoding sentences...")
        sentences = [m['sentence'] for m in self.mentions]
        sentence_embeddings = np.array([self.get_embedding(sent) for sent in sentences])
        
        print("Encoding documents...")
        doc_embeddings = []
        for mention_obj in self.mentions:
            doc_context = self.build_document_context(mention_obj['docid'])
            doc_emb = self.get_embedding(doc_context)
            doc_embeddings.append(doc_emb)
        doc_embeddings = np.array(doc_embeddings)
        
        # Normalize all embeddings
        mention_embeddings = mention_embeddings / np.linalg.norm(mention_embeddings, axis=1, keepdims=True)
        sentence_embeddings = sentence_embeddings / np.linalg.norm(sentence_embeddings, axis=1, keepdims=True)
        doc_embeddings = doc_embeddings / np.linalg.norm(doc_embeddings, axis=1, keepdims=True)
        
        # Combine with weights
        print("Combining three-level embeddings...")
        combined_embeddings = (mention_weight * mention_embeddings + 
                              sentence_weight * sentence_embeddings +
                              document_weight * doc_embeddings)
        
        print("Clustering...")
        clustering = AgglomerativeClustering(
            n_clusters=None,
            distance_threshold=distance_threshold,
            linkage=linkage,
            metric='cosine'
        )
        labels = clustering.fit_predict(combined_embeddings)
        
        return self._labels_to_clusters(labels)
    
    def two_stage_with_document_context(self,
                                       string_threshold=0.85,
                                       embedding_threshold=0.6,
                                       mention_weight=0.5,
                                       document_weight=0.5):
        """
        Method 4: Two-stage approach (your winning baseline + document context).
        
        Stage 1: Group by normalized mention string
        Stage 2: Merge clusters using mention+document embeddings
        
        This combines the speed of your baseline with document-aware semantics.
        """
        print("Stage 1: String-based grouping...")
        initial_clusters = defaultdict(list)
        for mention_obj in self.mentions:
            key = self.normalize_mention(mention_obj['mention'])
            initial_clusters[key].append(mention_obj['mention_id'])
        
        cluster_keys = list(initial_clusters.keys())
        print(f"Initial clusters: {len(cluster_keys)}")
        
        print("Stage 2: Computing cluster embeddings...")
        cluster_data = []
        
        for key in cluster_keys:
            mention_ids = initial_clusters[key]
            
            # Get mention embedding
            mention_emb = self.get_embedding(key)
            
            # Get average document context embedding for this cluster
            doc_embeddings = []
            for mid in mention_ids[:5]:  # Sample up to 5 for efficiency
                doc_context = self.build_document_context(self.mention_dict[mid]['docid'])
                doc_embeddings.append(self.get_embedding(doc_context))
            
            avg_doc_emb = np.mean(doc_embeddings, axis=0)
            
            cluster_data.append({
                'key': key,
                'mention_ids': mention_ids,
                'mention_emb': mention_emb,
                'doc_emb': avg_doc_emb
            })
        
        # Normalize and combine embeddings
        print("Combining embeddings...")
        combined_embeddings = []
        for cluster in cluster_data:
            mention_emb = cluster['mention_emb'] / np.linalg.norm(cluster['mention_emb'])
            doc_emb = cluster['doc_emb'] / np.linalg.norm(cluster['doc_emb'])
            combined = mention_weight * mention_emb + document_weight * doc_emb
            combined_embeddings.append(combined)
        
        combined_embeddings = np.array(combined_embeddings)
        
        # Compute similarity and merge
        print("Computing similarities and merging...")
        similarity_matrix = cosine_similarity(combined_embeddings)
        
        cluster_mapping = {}
        final_clusters = []
        next_cluster_id = 0
        
        for i in range(len(cluster_data)):
            if i in cluster_mapping:
                continue
            
            cluster_mapping[i] = next_cluster_id
            merged_mentions = cluster_data[i]['mention_ids'].copy()
            
            for j in range(i + 1, len(cluster_data)):
                if j not in cluster_mapping:
                    if similarity_matrix[i, j] >= embedding_threshold:
                        cluster_mapping[j] = next_cluster_id
                        merged_mentions.extend(cluster_data[j]['mention_ids'])
            
            final_clusters.append(merged_mentions)
            next_cluster_id += 1
        
        print(f"Final clusters: {len(final_clusters)}")
        return final_clusters
    
    def _labels_to_clusters(self, labels):
        """Convert cluster labels to list of mention_id lists."""
        clusters_dict = defaultdict(list)
        for idx, label in enumerate(labels):
            mention_id = self.mentions[idx]['mention_id']
            clusters_dict[label].append(mention_id)
        return list(clusters_dict.values())
    
    def save_clusters(self, clusters, filename='clusters.json'):
        """Save clusters to JSON file."""
        with open(filename, 'w') as f:
            json.dump(clusters, f, indent=2)
        print(f"Saved {len(clusters)} clusters to {filename}")


In [ ]:
print("="*70)
print("MENTION + DOCUMENT CONTEXT CLUSTERING")
print("="*70)

# Initialize
doc_embed = MentionDocumentEmbedding(
    test_data,
    embedding_model='all-MiniLM-L6-v2'  # Or try 'allenai/scibert_scivocab_uncased'
)

# # Method 1: Combined text embedding
# print("\n[Method 1] Mention + Document Combined Text...")
# clusters_combined = doc_embed.mention_plus_document_clustering(
#     distance_threshold=0.4,
#     mention_weight=0.5,
#     document_weight=0.5
# )
# print(f"Generated {len(clusters_combined)} clusters\n")

# Method 2: Separate embeddings
print("[Method 2] Separate Mention & Document Embeddings...")
clusters_separate = doc_embed.separate_embedding_clustering(
    distance_threshold=0.4,
    mention_weight=0.6,
    document_weight=0.4
)
print(f"Generated {len(clusters_separate)} clusters\n")

# # Method 3: Three-level embedding
# print("[Method 3] Three-Level (Mention + Sentence + Document)...")
# clusters_three_level = doc_embed.mention_sentence_document_clustering(
#     distance_threshold=0.4,
#     mention_weight=0.4,
#     sentence_weight=0.3,
#     document_weight=0.3
# )
# print(f"Generated {len(clusters_three_level)} clusters\n")

# # Method 4: Two-stage with document context (RECOMMENDED)
# print("[Method 4] Two-Stage with Document Context (RECOMMENDED)...")
# clusters_two_stage = doc_embed.two_stage_with_document_context(
#     embedding_threshold=0.6,
#     mention_weight=0.5,
#     document_weight=0.5
# )
# print(f"Generated {len(clusters_two_stage)} clusters\n")

# Save best method
print("="*70)
doc_embed.save_clusters(clusters_separate, 'clusters_subtask_2.json')


MENTION + DOCUMENT CONTEXT CLUSTERING
Loading embedding model: all-MiniLM-L6-v2...
Model loaded! Processing 12516 mentions from 1977 documents
[Method 2] Separate Mention & Document Embeddings...
Encoding mentions...
Encoding document contexts...
Combining embeddings...
Clustering...
Generated 1774 clusters

Saved 1774 clusters to clusters_subtask_2.json
